In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\| **Quickstart** \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Quickstart
==========

This section runs through the API for common tasks in machine learning.
Refer to the links in each section to dive deeper.

Working with data
-----------------

PyTorch has two [primitives to work with
data](https://pytorch.org/docs/stable/data.html):
`torch.utils.data.DataLoader` and `torch.utils.data.Dataset`. `Dataset`
stores the samples and their corresponding labels, and `DataLoader`
wraps an iterable around the `Dataset`.


In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

PyTorch offers domain-specific libraries such as
[TorchText](https://pytorch.org/text/stable/index.html),
[TorchVision](https://pytorch.org/vision/stable/index.html), and
[TorchAudio](https://pytorch.org/audio/stable/index.html), all of which
include datasets. For this tutorial, we will be using a TorchVision
dataset.

The `torchvision.datasets` module contains `Dataset` objects for many
real-world vision data like CIFAR, COCO ([full list
here](https://pytorch.org/vision/stable/datasets.html)). In this
tutorial, we use the FashionMNIST dataset. Every TorchVision `Dataset`
includes two arguments: `transform` and `target_transform` to modify the
samples and labels respectively.


In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 270kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 3.08MB/s]


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


We pass the `Dataset` as an argument to `DataLoader`. This wraps an
iterable over our dataset, and supports automatic batching, sampling,
shuffling and multiprocess data loading. Here we define a batch size of
64, i.e. each element in the dataloader iterable will return a batch of
64 features and labels.


In [4]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


Read more about [loading data in PyTorch](data_tutorial.html).


------------------------------------------------------------------------


Creating Models
===============

To define a neural network in PyTorch, we create a class that inherits
from
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
We define the layers of the network in the `__init__` function and
specify how data will pass through the network in the `forward`
function. To accelerate operations in the neural network, we move it to
the
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [14]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Read more about [building neural networks in
PyTorch](buildmodel_tutorial.html).


------------------------------------------------------------------------


Optimizing the Model Parameters
===============================

To train a model, we need a [loss
function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an
[optimizer](https://pytorch.org/docs/stable/optim.html).


In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training
dataset (fed to it in batches), and backpropagates the prediction error
to adjust the model\'s parameters.


In [16]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model\'s performance against the test dataset to
ensure it is learning.


In [17]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The training process is conducted over several iterations (*epochs*).
During each epoch, the model learns parameters to make better
predictions. We print the model\'s accuracy and loss at each epoch;
we\'d like to see the accuracy increase and the loss decrease with every
epoch.


In [18]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.322511  [   64/60000]
loss: 2.301809  [ 6464/60000]
loss: 2.279908  [12864/60000]
loss: 2.261220  [19264/60000]
loss: 2.249818  [25664/60000]
loss: 2.228192  [32064/60000]
loss: 2.226644  [38464/60000]
loss: 2.198589  [44864/60000]
loss: 2.197635  [51264/60000]
loss: 2.163149  [57664/60000]
Test Error: 
 Accuracy: 41.2%, Avg loss: 2.157654 

Epoch 2
-------------------------------
loss: 2.179691  [   64/60000]
loss: 2.165064  [ 6464/60000]
loss: 2.110553  [12864/60000]
loss: 2.112797  [19264/60000]
loss: 2.069353  [25664/60000]
loss: 2.022686  [32064/60000]
loss: 2.037685  [38464/60000]
loss: 1.968802  [44864/60000]
loss: 1.970796  [51264/60000]
loss: 1.898523  [57664/60000]
Test Error: 
 Accuracy: 57.0%, Avg loss: 1.895454 

Epoch 3
-------------------------------
loss: 1.936227  [   64/60000]
loss: 1.903318  [ 6464/60000]
loss: 1.790767  [12864/60000]
loss: 1.812264  [19264/60000]
loss: 1.711785  [25664/60000]
loss: 1.673368  [32064/600

Read more about [Training your model](optimization_tutorial.html).


------------------------------------------------------------------------


Saving Models
=============

A common way to save a model is to serialize the internal state
dictionary (containing the model parameters).


In [19]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


Loading Models
==============

The process for loading a model includes re-creating the model structure
and loading the state dictionary into it.


In [20]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

This model can now be used to make predictions.


In [21]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


### Accessing Downloaded Data Files

To see the files that the `FashionMNIST` dataset downloaded and that the original `model` was trained on, you can list the contents of the `data` directory.

In [23]:
import os

# List the contents of the 'data' directory
print("Contents of the 'data' directory:")
for root, dirs, files in os.walk('data'):
    level = root.replace('data', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

# You can also specifically look inside the FashionMNIST directory
print("\nContents of the 'data/FashionMNIST' directory:")
print(os.listdir('data/FashionMNIST'))

Contents of the 'data' directory:
data/
    FashionMNIST/
        raw/
            train-images-idx3-ubyte.gz
            train-images-idx3-ubyte
            t10k-labels-idx1-ubyte.gz
            t10k-images-idx3-ubyte
            train-labels-idx1-ubyte
            train-labels-idx1-ubyte.gz
            t10k-images-idx3-ubyte.gz
            t10k-labels-idx1-ubyte

Contents of the 'data/FashionMNIST' directory:
['raw']


Read more about [Saving & Loading your
model](saveloadrun_tutorial.html).


## Applying Machine Learning to Agriculture: Nutrition to Plant Growth

Let's apply these machine learning concepts to your agricultural scenario: predicting plant growth from nutrition factors. This is a *regression* problem, meaning we want to predict continuous numbers (like plant height, weight, etc.), not categories (like 'apple' or 'orange').

### 1. Defining Our Plant Growth Prediction Model

First, we need a special kind of 'brain' (our neural network) that can learn from the nutrition data and predict growth. Since we have 5 nutrition inputs and want 5 growth outputs, our model will be designed for that.

In [11]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Define the neural network for plant growth prediction
class PlantGrowthPredictor(nn.Module):
    def __init__(self, input_features=5, output_features=5):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_features, 64),  # First layer: 5 nutrition inputs -> 64 internal connections
            nn.ReLU(),                     # Activation function: adds non-linearity
            nn.Linear(64, 64),             # Second layer: 64 internal connections -> 64 more internal connections
            nn.ReLU(),                     # Another activation function
            nn.Linear(64, output_features) # Output layer: 64 internal connections -> 5 plant growth outputs
        )

    def forward(self, x):
        # Data flows through the layers defined above
        logits = self.linear_relu_stack(x)
        return logits

# Instantiate our new model
# We'll re-detect the device to ensure we're using the best available one
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

plant_model = PlantGrowthPredictor(input_features=5, output_features=5).to(device)
print(plant_model)

Using cpu device
PlantGrowthPredictor(
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)


#### Simple Explanation of the `PlantGrowthPredictor` Model:

Imagine your model is a small factory that takes raw ingredients (nutrition factors) and processes them to produce finished products (plant growth responses).

*   **`__init__` (Building the factory):** This is where you design the assembly line. We're setting up a series of processing stations (`nn.Linear` layers) and quality control checks (`nn.ReLU` activation functions).
    *   `nn.Linear(input_features, 64)`: The first station takes your 5 nutrition numbers and mixes them up in 64 different ways. These are called 'neurons' or 'nodes'.
    *   `nn.ReLU()`: This is a simple 'gate' that lets positive values pass through and stops negative ones. It helps the factory learn more complex patterns than just simple mixing.
    *   We have a few more mixing and gating stations to process the information further.
    *   `nn.Linear(64, output_features)`: The last station takes the processed information and turns it into your 5 predicted plant growth numbers.

*   **`forward` (Running the factory):** This describes the exact path your nutrition data (`x`) takes through the factory, from the input station all the way to the output station.

### 2. Training Function for Our Agricultural Model

Now, let's create a training function. This is like teaching our factory how to produce good predictions by showing it examples of nutrition and the resulting plant growth.

In [12]:
def train_plant_model(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train() # Set the model to training mode
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 1. Compute prediction error (how wrong is the factory's guess?)
        pred = model(X.float()) # Ensure input is float
        loss = loss_fn(pred, y.float()) # Ensure target is float

        # 2. Backpropagation (figuring out who on the assembly line made a mistake)
        optimizer.zero_grad() # Clear previous gradients
        loss.backward()       # Calculate gradients
        optimizer.step()      # Adjust the assembly line based on mistakes

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

#### Simple Explanation of the `train_plant_model` Function:

This function describes one full practice session for our plant growth prediction factory.

*   **`model.train()`:** We tell the factory, "Okay, we're in practice mode now. Learn from your mistakes!"
*   **Loop through `dataloader` (Practice with small batches):** Instead of showing the factory *all* the data at once (which would be too much for its 'brain'), we show it small batches of nutrition factors and their *actual* corresponding plant growth. Each `X` is a batch of nutrition factors, and `y` is the batch of actual plant growth.
    *   **`pred = model(X.float())` (Make a guess):** The factory takes the nutrition factors (`X`) and makes its own guess (`pred`) about the plant growth.
    *   **`loss = loss_fn(pred, y.float())` (Check how wrong it is):** We use a 'teacher' (`loss_fn`) to compare the factory's guess (`pred`) with the *actual* plant growth (`y`). The `loss` tells us exactly how big the difference (the error) is.
        *   For regression (predicting numbers), a common 'teacher' is `nn.MSELoss()` (Mean Squared Error), which measures the average squared difference between the guess and the actual value. We want this number to be as small as possible!
    *   **`optimizer.zero_grad()` (Clear the whiteboard):** Before figuring out new mistakes, we erase all the old mistake calculations.
    *   **`loss.backward()` (Find out who made the mistake):** This is the magic part! The factory intelligently figures out exactly which parts of its internal mechanisms (parameters) contributed most to the `loss`.
    *   **`optimizer.step()` (Fix the mistakes):** Based on `loss.backward()`, the `optimizer` (our 'repair crew') makes tiny adjustments to the factory's internal mechanisms to make its next guess a little bit better. The `lr` (learning rate) tells the repair crew how big these adjustments should be. Too big, and they might break something; too small, and it takes forever.

This whole process repeats for many batches, and many 'epochs' (full passes through all the data), gradually making the factory's predictions more accurate.

### 3. Example Usage: Generating Dummy Data and Training

Since you don't have a dataset loaded yet, let's create some dummy data to show how it works. We'll simulate 5 nutrition factors (inputs) and 5 plant growth responses (outputs).

In [13]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# --- Create Dummy Data for Demonstration ---
# Imagine 1000 samples of nutrition data and corresponding growth data
num_samples = 1000
num_nutrition_factors = 5
num_growth_parameters = 5

# Random nutrition data (e.g., N, P, K, pH, sunlight hours)
dummy_nutrition_data = torch.randn(num_samples, num_nutrition_factors) * 10 + 50 # Example: centered around 50

# Random plant growth data (e.g., height, leaf count, biomass, yield, root length)
# Let's make growth somewhat dependent on nutrition, but with noise
dummy_growth_data = (dummy_nutrition_data * 0.5 + torch.randn(num_samples, num_growth_parameters) * 5) + 20

# Create a PyTorch Dataset and DataLoader
dummy_dataset = TensorDataset(dummy_nutrition_data, dummy_growth_data)
dummy_dataloader = DataLoader(dummy_dataset, batch_size=32, shuffle=True)

print(f"Dummy input data shape: {dummy_nutrition_data.shape}")
print(f"Dummy target data shape: {dummy_growth_data.shape}")

# --- Set up for training ---

# For regression tasks, Mean Squared Error (MSE) is a common choice
regression_loss_fn = nn.MSELoss()

# We can use a different optimizer, like Adam, which often performs well
regression_optimizer = torch.optim.Adam(plant_model.parameters(), lr=0.001)

epochs = 10
print("\nStarting training with dummy data...")
for t in range(epochs):
    print(f"Epoch {t+1}\n------------------------------- ")
    train_plant_model(dummy_dataloader, plant_model, regression_loss_fn, regression_optimizer, device)
print("Done training with dummy data!")

# You would typically also have a 'test' function for this model
# to evaluate its performance on unseen data, similar to the quickstart example.

Dummy input data shape: torch.Size([1000, 5])
Dummy target data shape: torch.Size([1000, 5])

Starting training with dummy data...
Epoch 1
------------------------------- 
loss: 1798.427124  [   32/ 1000]
Epoch 2
------------------------------- 
loss: 94.764114  [   32/ 1000]
Epoch 3
------------------------------- 
loss: 44.741798  [   32/ 1000]
Epoch 4
------------------------------- 
loss: 56.462425  [   32/ 1000]
Epoch 5
------------------------------- 
loss: 30.276520  [   32/ 1000]
Epoch 6
------------------------------- 
loss: 32.499538  [   32/ 1000]
Epoch 7
------------------------------- 
loss: 26.794317  [   32/ 1000]
Epoch 8
------------------------------- 
loss: 28.046240  [   32/ 1000]
Epoch 9
------------------------------- 
loss: 33.240253  [   32/ 1000]
Epoch 10
------------------------------- 
loss: 23.996166  [   32/ 1000]
Done training with dummy data!


### 4. Relating to Your 4 Key Understanding Points & Common Pitfalls

Let's revisit the four points we discussed and talk about common mistakes:

1.  **What problem the model solves?**
    *   **For you:** Predicting 5 plant growth parameters (continuous numbers) based on 5 nutrition factors (continuous numbers). This is a **multivariate regression** problem.
    *   **Pitfall:** Using a model or loss function designed for classification (like predicting 'disease' or 'no disease') when you need to predict numbers. We specifically chose `PlantGrowthPredictor` (with a linear output layer) and `nn.MSELoss` to avoid this.

2.  **What kind of data it needs?**
    *   **For you:** Numerical data for both nutrition factors (inputs, e.g., `% Nitrogen`, `ppm Phosphorus`) and plant growth responses (outputs, e.g., `cm Height`, `g Biomass`). Each row in your dataset should be one plant observation, with 5 nutrition numbers and 5 corresponding growth numbers.
    *   **Pitfall 1 (Data Quality):** "Garbage in, garbage out!" If your sensor readings are wrong, or your plant growth measurements are inconsistent, the model will learn those errors. Ensure your data is clean, accurate, and consistently measured.
    *   **Pitfall 2 (Data Quantity):** Neural networks often need a lot of data to learn well. With only a few plant samples, the model might not generalize to new plants or conditions.
    *   **Pitfall 3 (Data Scaling):** Imagine one nutrition factor is `pH` (range 0-14) and another is `Nitrogen` (range 0-1000 ppm). The model might think `Nitrogen` is much more important just because its numbers are bigger. We often `scale` (normalize) all input data to a similar range (e.g., between 0 and 1) so the model treats them fairly.

3.  **How to feed your data to it?**
    *   **For you:** Your data needs to be in a numerical format that PyTorch can understand (tensors). We used `TensorDataset` and `DataLoader` to organize our dummy data into batches, which is the standard way to feed data to a PyTorch model.
    *   **Pitfall 1 (Wrong Format):** Trying to feed text descriptions or images directly without converting them to numbers. Your `nutrition_data` and `growth_data` should be arrays of numbers.
    *   **Pitfall 2 (Incorrect Shapes):** If your model expects 5 inputs but you feed it 6, it will error out. Always check the `shape` of your data (`.shape` in PyTorch/NumPy) to match the model's design.

4.  **How to interpret its results?**
    *   **For you:** The `pred` from your `plant_model` will be 5 numbers, representing the model's best guess for the 5 plant growth parameters. The `loss` value (from `nn.MSELoss`) tells you how good these predictions are, with lower numbers being better. You'd also want to use a separate `test` dataset (like the quickstart's `test` function) to see how well your model performs on new, unseen plants.
    *   **Pitfall 1 (Overfitting):** The model becomes *too good* at predicting the plant growth from the training data, but fails miserably on new plants. It's like memorizing answers for a test without truly understanding the subject. You'll see low training loss but high test loss. This is why having a separate test set is crucial.
    *   **Pitfall 2 (Underfitting):** The model isn't powerful enough or hasn't trained long enough to learn the patterns in the data, leading to high loss on both training and test sets.
    *   **Pitfall 3 (Misinterpreting Output):** Just because the model gives you numbers doesn't mean they're good or meaningful. Always compare the predicted values to actual values and common sense for your agricultural domain.

### 5. Making Predictions with the Trained Agricultural Model

Now that our `PlantGrowthPredictor` has been trained (even on dummy data), we can use it to make predictions on new, unseen nutrition data. This is how the 'prediction and actual' phase works for a regression model like ours.

In [ ]:
# Put the model in evaluation mode
plant_model.eval()

# Create some new dummy nutrition data for prediction
# This simulates new plant observations where we only know nutrition factors
num_new_samples = 10
new_nutrition_data = torch.randn(num_new_samples, num_nutrition_factors) * 10 + 50

# For demonstration, let's also create corresponding 'actual' growth data
# In a real scenario, this 'actual' data would come from real measurements
new_actual_growth_data = (new_nutrition_data * 0.5 + torch.randn(num_new_samples, num_growth_parameters) * 5) + 20

print(f"New nutrition data shape: {new_nutrition_data.shape}")
print(f"New actual growth data shape: {new_actual_growth_data.shape}")

# Make predictions with the model
# We use torch.no_grad() because we don't need to calculate gradients for inference
with torch.no_grad():
    # Move input data to the same device as the model
    input_data_on_device = new_nutrition_data.float().to(device)
    predictions = plant_model(input_data_on_device)

# Move predictions and actuals back to CPU for easier comparison if they were on GPU
predictions_cpu = predictions.cpu()
actuals_cpu = new_actual_growth_data.cpu()

print("\n--- Predictions vs. Actuals ---")
for i in range(num_new_samples):
    print(f"Sample {i+1}:")
    print(f"  Nutrition Input: {new_nutrition_data[i].numpy()}")
    print(f"  Predicted Growth: {predictions_cpu[i].numpy()}")
    print(f"  Actual Growth:    {actuals_cpu[i].numpy()}")
    print("-------------------------------")

# Calculate a performance metric for the predictions
# For regression, Mean Squared Error (MSE) is commonly used
# We'll use the same loss function defined earlier
mse_on_new_data = regression_loss_fn(predictions_cpu, actuals_cpu.float())
print(f"Mean Squared Error on new data: {mse_on_new_data.item():>8f}")

# A lower MSE indicates better prediction accuracy. If you trained for more epochs,
# this MSE would likely be lower (closer to the training loss).

### Explanation of the 'Prediction and Actual' Phase:

1.  **`plant_model.eval()`:** Before making predictions, it's crucial to set the model to evaluation mode. This disables certain layers like Dropout or BatchNorm that behave differently during training and inference, ensuring consistent predictions.
2.  **New Data Preparation:** We create `new_nutrition_data` to represent fresh, unseen inputs for our model. In a real application, this would be new sensor readings or experimental conditions.
3.  **`torch.no_grad()`:** When we make predictions, we're not trying to optimize the model's parameters, so we don't need to compute or store gradients. `torch.no_grad()` saves memory and speeds up computations.
4.  **`predictions = plant_model(input_data_on_device)`:** We pass the new input data through our trained `plant_model`. The model processes the nutrition factors and outputs its best guess for the corresponding plant growth parameters.
5.  **Comparison and Evaluation:**
    *   We print the input nutrition, the model's `predictions`, and the `actual` growth data (which we generated for this demonstration). This visual comparison helps to see how close the predictions are to reality.
    *   We then calculate the `Mean Squared Error (MSE)` between the `predictions` and the `actuals` using `regression_loss_fn`. This gives us a single numerical value quantifying the overall error of our model on this new data. A lower MSE indicates better performance.

This process is fundamental for understanding how well your machine learning model generalizes to data it has never seen before, which is the ultimate goal of any predictive model.

### 6. Generating a Downloadable CSV for New Nutrition Data

Let's create a CSV file with hypothetical new nutrition data that you could use as input for your `PlantGrowthPredictor` model. This allows you to easily prepare new scenarios for prediction.

In [24]:
import pandas as pd
from google.colab import files

# Define the number of new samples for our fake scenario
num_prediction_samples = 500

# Generate new dummy nutrition data (e.g., N, P, K, pH, sunlight hours)
# We'll use the same generation logic as before for consistency
further_new_nutrition_data = torch.randn(num_prediction_samples, num_nutrition_factors) * 10 + 50

# Convert the Torch tensor to a Pandas DataFrame
nutrition_df = pd.DataFrame(
    further_new_nutrition_data.numpy(),
    columns=[f'Nutrition_Factor_{i+1}' for i in range(num_nutrition_factors)]
)

# Display the first few rows of the DataFrame
print("Generated Nutrition Data for Prediction:")
display(nutrition_df.head())

# Define the filename for the CSV
csv_filename = 'new_nutrition_data_for_prediction.csv'

# Save the DataFrame to a CSV file
nutrition_df.to_csv(csv_filename, index=False)

print(f"\n'{csv_filename}' created successfully. You can download it below:")

# Provide a download link for the user
files.download(csv_filename)


Generated Nutrition Data for Prediction:


,Nutrition_Factor_1,Nutrition_Factor_2,Nutrition_Factor_3,Nutrition_Factor_4,Nutrition_Factor_5
0,59.574757,59.105431,43.687660,44.459270,77.626663
1,52.154766,50.613697,48.157894,42.625393,55.475746
2,53.284992,49.068726,47.521339,56.609951,49.705502
3,36.917873,48.542183,57.557041,41.514435,64.747383
4,53.387211,43.383354,62.137497,44.143013,40.908844



'new_nutrition_data_for_prediction.csv' created successfully. You can download it below:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 7. Making Predictions with the Generated Nutrition Data

Now, let's use the `nutrition_df` (the data we just generated and saved to CSV) to make predictions with our trained `PlantGrowthPredictor` model.

In [26]:
# Ensure the model is in evaluation mode
plant_model.eval()

# Convert the pandas DataFrame to a PyTorch tensor
# Ensure the data type matches what the model expects (float)
prediction_input_tensor = torch.tensor(nutrition_df.values, dtype=torch.float32)

print(f"Input tensor shape for prediction: {prediction_input_tensor.shape}")

# Make predictions with the model
# Use torch.no_grad() for inference to save memory and computation
with torch.no_grad():
    # Move the input tensor to the same device as the model
    input_data_on_device = prediction_input_tensor.to(device)

    # Get predictions
    predictions_on_df = plant_model(input_data_on_device)

# Move predictions back to CPU for easier handling and display
predictions_on_df_cpu = predictions_on_df.cpu()

# Convert predictions to a pandas DataFrame for better readability
predicted_growth_df = pd.DataFrame(
    predictions_on_df_cpu.numpy(),
    columns=[f'Predicted_Growth_{i+1}' for i in range(num_growth_parameters)]
)

print("\n--- Model Predictions on New Nutrition Data ---")
display(predicted_growth_df.head())

# Optionally, you can concatenate the input nutrition data with the predictions
# to see the full picture
full_prediction_output_df = pd.concat([nutrition_df, predicted_growth_df], axis=1)
print("\n--- Combined Nutrition Input and Predicted Growth (First 5 Samples) ---")
display(full_prediction_output_df.head())

Input tensor shape for prediction: torch.Size([500, 5])

--- Model Predictions on New Nutrition Data ---


,Predicted_Growth_1,Predicted_Growth_2,Predicted_Growth_3,Predicted_Growth_4,Predicted_Growth_5
0,51.990200,52.478840,44.496094,45.674648,60.924099
1,45.560390,45.023640,43.584846,41.057606,47.162701
2,46.322975,44.584717,43.205162,48.281300,44.174404
3,37.996643,43.890137,48.920616,40.142387,51.831562
4,45.807842,40.815865,50.147381,41.429226,39.512665



--- Combined Nutrition Input and Predicted Growth (First 5 Samples) ---


,Nutrition_Factor_1,Nutrition_Factor_2,Nutrition_Factor_3,Nutrition_Factor_4,Nutrition_Factor_5,Predicted_Growth_1,Predicted_Growth_2,Predicted_Growth_3,Predicted_Growth_4,Predicted_Growth_5
0,59.574757,59.105431,43.687660,44.459270,77.626663,51.990200,52.478840,44.496094,45.674648,60.924099
1,52.154766,50.613697,48.157894,42.625393,55.475746,45.560390,45.023640,43.584846,41.057606,47.162701
2,53.284992,49.068726,47.521339,56.609951,49.705502,46.322975,44.584717,43.205162,48.281300,44.174404
3,36.917873,48.542183,57.557041,41.514435,64.747383,37.996643,43.890137,48.920616,40.142387,51.831562
4,53.387211,43.383354,62.137497,44.143013,40.908844,45.807842,40.815865,50.147381,41.429226,39.512665


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [27]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

print("--- Full Model Predictions on New Nutrition Data (Interactive Table) ---")
display(full_prediction_output_df)

--- Full Model Predictions on New Nutrition Data (Interactive Table) ---


,Nutrition_Factor_1,Nutrition_Factor_2,Nutrition_Factor_3,Nutrition_Factor_4,Nutrition_Factor_5,Predicted_Growth_1,Predicted_Growth_2,Predicted_Growth_3,Predicted_Growth_4,Predicted_Growth_5
0,59.574757,59.105431,43.687660,44.459270,77.626663,51.990200,52.478840,44.496094,45.674648,60.924099
1,52.154766,50.613697,48.157894,42.625393,55.475746,45.560390,45.023640,43.584846,41.057606,47.162701
2,53.284992,49.068726,47.521339,56.609951,49.705502,46.322975,44.584717,43.205162,48.281300,44.174404
3,36.917873,48.542183,57.557041,41.514435,64.747383,37.996643,43.890137,48.920616,40.142387,51.831562
4,53.387211,43.383354,62.137497,44.143013,40.908844,45.807842,40.815865,50.147381,41.429226,39.512665
...,...,...,...,...,...,...,...,...,...,...
495,56.833344,45.316525,47.762394,52.232872,65.229904,49.126640,43.833847,44.096001,47.331863,53.413967
496,29.604584,55.922165,45.527836,41.521603,49.658634,32.095806,45.154442,41.241104,37.607174,41.852497
497,30.126741,52.028561,59.761673,57.732502,59.866680,35.257168,45.709949,50.464096,47.820946,50.137753
498,37.041996,53.317863,75.103928,38.883301,37.604813,37.915424,45.635006,57.307728,37.938656,37.817375


### Explanation of the Downloadable CSV Generation:

1.  **Generate Data:** We create a new set of dummy nutrition data using `torch.randn` to simulate varied inputs, similar to how we created the previous `new_nutrition_data`.
2.  **Pandas DataFrame:** The generated Torch tensor is converted into a Pandas DataFrame. This is a standard and convenient way to handle tabular data in Python, especially for saving to CSV.
3.  **Column Names:** We assign generic column names like `Nutrition_Factor_1`, `Nutrition_Factor_2`, etc., to make the CSV file easily understandable.
4.  **Save to CSV:** `nutrition_df.to_csv(csv_filename, index=False)` saves the DataFrame to a file named `new_nutrition_data_for_prediction.csv`. `index=False` prevents Pandas from writing the DataFrame index as a column in the CSV.
5.  **Download Link:** `files.download(csv_filename)` is a Colab-specific function that provides a direct download link for the generated file, making it easy for you to access it locally.